In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from transformers import EarlyStoppingCallback

In [2]:
import tensorflow as tf

# Check available GPUs
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("GPUs are available:")
    for gpu in gpus:
        print(gpu)
else:
    print("No GPUs detected.")


2024-12-24 07:17:33.514012: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-24 07:17:33.514136: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-24 07:17:33.662597: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


GPUs are available:
PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [3]:
#clear the cache
tf.keras.backend.clear_session()

In [4]:
ds_path = Path('/kaggle/input/cottonweedid15/CottonWeedID15').resolve()

In [5]:
ds_path

PosixPath('/kaggle/input/cottonweedid15/CottonWeedID15')

In [6]:
paths = list(ds_path.glob('*/*'))
classes = [path.parent.stem for path in paths]

In [7]:
df = pd.DataFrame({'img':paths, 'class': classes})
df['class'] = df['class'].astype('category')
df['label'] = df['class'].cat.codes
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5187 entries, 0 to 5186
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   img     5187 non-null   object  
 1   class   5187 non-null   category
 2   label   5187 non-null   int8    
dtypes: category(1), int8(1), object(1)
memory usage: 51.4+ KB


In [8]:
class_labels = dict(zip(range(15), df['class'].cat.categories))
class_labels

{0: 'Carpetweeds',
 1: 'Crabgrass',
 2: 'Eclipta',
 3: 'Goosegrass',
 4: 'Morningglory',
 5: 'Nutsedge',
 6: 'PalmerAmaranth',
 7: 'Prickly Sida',
 8: 'Purslane',
 9: 'Ragweed',
 10: 'Sicklepod',
 11: 'SpottedSpurge',
 12: 'SpurredAnoda',
 13: 'Swinecress',
 14: 'Waterhemp'}

In [9]:
num_classes = len(class_labels)
num_classes

15

In [10]:
df['img']=df['img'].astype(str)
#df['label']=df['img'].astype(str)

In [11]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.3, shuffle=True, random_state=1230, stratify=df['label'])
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
len(train), len(test)

(3630, 1557)

In [12]:
train.reset_index(inplace=True)
test.reset_index(inplace=True)

In [13]:
train.head()

,index,img,class,label
0,0,/kaggle/input/cottonweedid15/CottonWeedID15/Wa...,Waterhemp,14
1,1,/kaggle/input/cottonweedid15/CottonWeedID15/Ca...,Carpetweeds,0
2,2,/kaggle/input/cottonweedid15/CottonWeedID15/Pa...,PalmerAmaranth,6
3,3,/kaggle/input/cottonweedid15/CottonWeedID15/Mo...,Morningglory,4
4,4,/kaggle/input/cottonweedid15/CottonWeedID15/Ca...,Carpetweeds,0


In [14]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    validation_split=0.1  # Set validation split
)

In [15]:
image_size = (224, 224)  # Change based on your model's requirements
batch_size = 32

In [16]:
# Train Data Generator
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train,
    x_col='img',
    y_col='class',
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'  # Use subset for training
)

Found 3267 validated image filenames belonging to 15 classes.


In [17]:
print("Class indices:", train_generator.class_indices)

Class indices: {'Carpetweeds': 0, 'Crabgrass': 1, 'Eclipta': 2, 'Goosegrass': 3, 'Morningglory': 4, 'Nutsedge': 5, 'PalmerAmaranth': 6, 'Prickly Sida': 7, 'Purslane': 8, 'Ragweed': 9, 'Sicklepod': 10, 'SpottedSpurge': 11, 'SpurredAnoda': 12, 'Swinecress': 13, 'Waterhemp': 14}


In [18]:
# Validation Data Generator
validation_generator = train_datagen.flow_from_dataframe(
    dataframe=train,
    x_col='img',
    y_col='class',
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'  # Use subset for validation
)

Found 363 validated image filenames belonging to 15 classes.


In [19]:
print("Class indices:", validation_generator.class_indices)

Class indices: {'Carpetweeds': 0, 'Crabgrass': 1, 'Eclipta': 2, 'Goosegrass': 3, 'Morningglory': 4, 'Nutsedge': 5, 'PalmerAmaranth': 6, 'Prickly Sida': 7, 'Purslane': 8, 'Ragweed': 9, 'Sicklepod': 10, 'SpottedSpurge': 11, 'SpurredAnoda': 12, 'Swinecress': 13, 'Waterhemp': 14}


In [20]:
# Create ImageDataGenerator for test (no augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)

In [21]:
# Test Data Generator
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test,
    x_col='img',
    y_col='class',  # Assuming test data might still have labels, if not, use None
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',  # Use 'None' if no labels are available
    shuffle=False  # Important for test data to maintain order
)

Found 1557 validated image filenames belonging to 15 classes.


In [22]:
from transformers import ViTForImageClassification, ViTFeatureExtractor, Trainer, TrainingArguments
from torch.utils.data import Dataset
# Load the ViT feature extractor and model
feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224-in21k')
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224-in21k', num_labels=num_classes)

# Convert the data to PyTorch-friendly format (using DataLoader)
class ImageClassificationDataset(Dataset):
    def __init__(self, generator, feature_extractor):
        self.generator = generator
        self.feature_extractor = feature_extractor
        self.image_paths = self.generator.filepaths
        self.labels = self.generator.labels
        self.classes = self.generator.class_indices
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Load and preprocess image using the ViT feature extractor
        image = tf.keras.preprocessing.image.load_img(image_path, target_size=(224, 224))
        image = tf.keras.preprocessing.image.img_to_array(image)
        
        # Normalize and prepare for ViT
        inputs = self.feature_extractor(images=image, return_tensors='pt')
        
        # Return the prepared image and label as PyTorch tensor
        return {'pixel_values': inputs['pixel_values'].squeeze(), 'labels': torch.tensor(label)}

# Create PyTorch datasets and dataloaders
train_dataset = ImageClassificationDataset(train_generator, feature_extractor)

val_dataset = ImageClassificationDataset(validation_generator, feature_extractor)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/models/vit/feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [24]:
from torch import nn
import tensorflow as tf
from tqdm import tqdm

# Check if GPU is available and assign the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define TrainingArguments for Hugging Face Trainer
training_args = TrainingArguments(
    output_dir='./results',               # Output directory for checkpoints and logs
    save_strategy="epoch",                # Save the model every epoch
    logging_dir='./logs',                 # Directory for logs
    logging_steps=10,                     # Log every 10 steps
    logging_first_step=True,              # Log the first step
    load_best_model_at_end=True,          # Load the best model at the end
    metric_for_best_model="eval_loss",     # Metric to evaluate the best model
    greater_is_better=False,               # We want higher accuracy
    num_train_epochs=20,                   # Number of epochs
    per_device_train_batch_size=batch_size,  # Batch size per device during training
    per_device_eval_batch_size=batch_size,   # Batch size per device during evaluation
    eval_strategy="epoch",          # Evaluation at the end of each epoch
    save_steps=10,                        # Save checkpoints every 10 steps
    save_total_limit=1,                    # Keep only the best checkpoint (1 model)   
    fp16=True,                            # Use FP16 for faster training (if supported)
    disable_tqdm=True,                   # Enable tqdm progress bar
)

# Add EarlyStoppingCallback to the Trainer
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=4,  # Stop training if no improvement for 3 evaluation steps
    early_stopping_threshold=0.01  # Minimum improvement for considering an improvement
)

# Define the Trainer
trainer = Trainer(
    model=model,                          # The model to train
    args=training_args,                   # Training arguments
    train_dataset=train_dataset,          # Training dataset
    eval_dataset=val_dataset,             # Validation dataset
    tokenizer=feature_extractor,          # Feature extractor for preprocessing
    callbacks=[early_stopping_callback]
)

# Start training
trainer.train()

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


{'loss': 2.6954, 'grad_norm': 1.7061498165130615, 'learning_rate': 4.9975728155339805e-05, 'epoch': 0.009708737864077669}
{'loss': 2.4924, 'grad_norm': 1.7810664176940918, 'learning_rate': 4.975728155339806e-05, 'epoch': 0.0970873786407767}
{'loss': 2.1961, 'grad_norm': 1.5261512994766235, 'learning_rate': 4.951456310679612e-05, 'epoch': 0.1941747572815534}
{'loss': 1.9398, 'grad_norm': 1.547436237335205, 'learning_rate': 4.9271844660194174e-05, 'epoch': 0.2912621359223301}
{'loss': 1.6817, 'grad_norm': 1.6310601234436035, 'learning_rate': 4.902912621359224e-05, 'epoch': 0.3883495145631068}
{'loss': 1.4534, 'grad_norm': 1.7039722204208374, 'learning_rate': 4.8786407766990296e-05, 'epoch': 0.4854368932038835}
{'loss': 1.2326, 'grad_norm': 1.5529907941818237, 'learning_rate': 4.854368932038835e-05, 'epoch': 0.5825242718446602}
{'loss': 1.1445, 'grad_norm': 2.010887622833252, 'learning_rate': 4.830097087378641e-05, 'epoch': 0.6796116504854369}
{'loss': 0.9749, 'grad_norm': 1.4622434377670

TrainOutput(global_step=1030, training_loss=0.2664249513045098, metrics={'train_runtime': 2750.8903, 'train_samples_per_second': 23.752, 'train_steps_per_second': 0.749, 'train_loss': 0.2664249513045098, 'epoch': 10.0})

In [25]:
test_dataset = ImageClassificationDataset(test_generator, feature_extractor)

In [26]:
### from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, classification_report
predictions, labels, _ = trainer.predict(test_dataset)

# Convert logits to predicted labels
preds = np.argmax(predictions, axis=1)

# Now, calculate the classification report
report = classification_report(labels, preds, digits=4)

# Print the classification report
print("Classification Report:\n", report)

Classification Report:
               precision    recall  f1-score   support

           0     0.9826    0.9869    0.9847       229
           1     1.0000    0.9697    0.9846        33
           2     0.8929    0.9868    0.9375        76
           3     0.9846    0.9846    0.9846        65
           4     0.9880    0.9851    0.9865       335
           5     1.0000    1.0000    1.0000        82
           6     0.9668    0.9855    0.9761       207
           7     0.9737    0.9487    0.9610        39
           8     0.9850    0.9704    0.9776       135
           9     0.9750    1.0000    0.9873        39
          10     0.9861    0.9861    0.9861        72
          11     1.0000    1.0000    1.0000        70
          12     1.0000    0.7778    0.8750        18
          13     1.0000    0.9545    0.9767        22
          14     0.9847    0.9556    0.9699       135

    accuracy                         0.9794      1557
   macro avg     0.9813    0.9661    0.9725      1557
we